# DriveSense-VLM — 00: Data Pipeline (Full Scale)

**GPU**: T4 (or CPU) | **Time**: ~3–4 h | **Cost**: ~8 CU + $15–20 Claude API

> ⚠️ **Use a FRESH runtime.** Never downgrade numpy — Colab ships numpy 2.x.
> Set `FORCE_RERUN = True` to rebuild any step from scratch.

Pipeline:
1. **Phase A** — nuScenes trainval download (850 scenes, 28 K+ keyframes, ~350 GB)
2. **Phase B** — DADA-2000 download from HuggingFace (~15 GB via git lfs)
3. **Phase C** — Rarity filtering (score ≥ 3) → 500–800 frames
4. **Phase D** — DADA-2000 critical-moment extraction → 200–400 frames
5. **Phase E** — PySpark distributed ETL + unified train/val/test splits
6. **Phase F** — LLM annotation (`USE_MOCK_LLM=False` → real Claude API, ~$15–20)
7. **Phase G** — SFT JSONL formatting with correct absolute image paths

**Target**: 700–1,200 high-quality diverse SFT examples for production training.

In [ ]:
# ══════════════════════════════════════════════════════════
# CONFIGURATION — Edit these before running
# ══════════════════════════════════════════════════════════
FORCE_RERUN      = True    # True: always rerun each step (overwrite outputs)
                            # False: skip a step if its output already exists
USE_MOCK_LLM     = False   # True: free mock annotations (no API key)
                            # False: real Claude API (~$15-20, needs ANTHROPIC_API_KEY)
NUSCENES_VERSION = "v1.0-trainval"
MIN_RARITY_SCORE = 3       # Keep frames with rarity score >= 3 (out of 6)
DOWNLOAD_DADA    = True    # True: clone DADA-2000 from HuggingFace (~15 GB)
                            # False: skip if already on Drive or not needed
# ══════════════════════════════════════════════════════════

In [ ]:
# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil

# ── Paths ──
PROJECT_ROOT = "/content/drive/MyDrive/DriveSense-VLM"
REPO_ROOT    = "/content/DriveSense-VLM"
DATA_ROOT    = f"{PROJECT_ROOT}/data"
OUTPUTS_ROOT = f"{PROJECT_ROOT}/outputs"
LOCAL_DATA   = "/content/raw_data"   # ephemeral SSD for large downloads

# Create Drive directories
for d in [DATA_ROOT, f"{DATA_ROOT}/nuscenes", f"{DATA_ROOT}/dada2000",
          OUTPUTS_ROOT, f"{OUTPUTS_ROOT}/data", f"{OUTPUTS_ROOT}/training",
          f"{OUTPUTS_ROOT}/data/sft_ready"]:
    os.makedirs(d, exist_ok=True)
os.makedirs(LOCAL_DATA, exist_ok=True)

# Clone repo to fast ephemeral SSD (skip if already cloned — saves bandwidth)
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/jayanth922/DriveSense-VLM.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
os.chdir(REPO_ROOT)
print(f"Working dir: {os.getcwd()}")

# Symlink data/outputs → Drive (persistent across disconnects)
!ln -sfn {DATA_ROOT} {REPO_ROOT}/data
!ln -sfn {OUTPUTS_ROOT} {REPO_ROOT}/outputs

# Add src to Python path (replaces broken editable install)
sys.path.insert(0, f"{REPO_ROOT}/src")
print(f"✓ Repo:  {REPO_ROOT}")
print(f"✓ Drive: {PROJECT_ROOT}")

# Install data-pipeline dependencies
# ⚠️  NEVER install numpy or pandas — Colab pre-installs numpy 2.x / pandas 2.x.
#     Reinstalling them breaks ABI compatibility with pre-compiled extensions.
# nuscenes-devkit pins numpy<2 — install with --no-deps then add real runtime deps.
print("\nInstalling dependencies...")
!pip install --upgrade pip setuptools wheel -q 2>&1 | tail -1
!pip install nuscenes-devkit --no-deps -q 2>&1 | tail -1
!pip install pyquaternion matplotlib tqdm Pillow pyyaml requests openpyxl -q 2>&1 | tail -1
!pip install scikit-learn scipy pyspark -q 2>&1 | tail -1

import numpy as np, pandas as pd, drivesense
print(f"✓ numpy {np.__version__} | pandas {pd.__version__} | DriveSense loaded")
from nuscenes.nuscenes import NuScenes
print("✓ nuscenes-devkit importable")

# Disk space check — nuScenes trainval is ~350 GB total (but we delete tarballs on-the-fly)
!df -h /content | tail -1
print("Tip: Colab ephemeral SSD is ~200 GB. We extract+delete each tarball to stay within budget.")

## Phase A — nuScenes Trainval Download

nuScenes trainval is split into 11 compressed archives (~350 GB total).
**Strategy**: download → extract → delete each tarball immediately to stay within
Colab's ~200 GB SSD budget. Skip already-extracted files.

In [ ]:
import os

NUSCENES_ROOT = f"{DATA_ROOT}/nuscenes"
os.makedirs(NUSCENES_ROOT, exist_ok=True)

URLS = [
    ("v1.0-trainval_meta.tgz",
     "https://d36yt3mvayqw5m.cloudfront.net/public/v1.0/v1.0-trainval_meta.tgz"),
    ("v1.0-trainval01_blobs.tgz",
     "https://motional-nuscenes.s3.amazonaws.com/public/v1.0/v1.0-trainval01_blobs.tgz"),
    ("v1.0-trainval02_blobs.tgz",
     "https://motional-nuscenes.s3.amazonaws.com/public/v1.0/v1.0-trainval02_blobs.tgz"),
    ("v1.0-trainval03_blobs.tgz",
     "https://d36yt3mvayqw5m.cloudfront.net/public/v1.0/v1.0-trainval03_blobs.tgz"),
    ("v1.0-trainval04_blobs.tgz",
     "https://motional-nuscenes.s3.amazonaws.com/public/v1.0/v1.0-trainval04_blobs.tgz"),
    ("v1.0-trainval05_blobs.tgz",
     "https://motional-nuscenes.s3.amazonaws.com/public/v1.0/v1.0-trainval05_blobs.tgz"),
    ("v1.0-trainval06_blobs.tgz",
     "https://d36yt3mvayqw5m.cloudfront.net/public/v1.0/v1.0-trainval06_blobs.tgz"),
    ("v1.0-trainval07_blobs.tgz",
     "https://d36yt3mvayqw5m.cloudfront.net/public/v1.0/v1.0-trainval07_blobs.tgz"),
    ("v1.0-trainval08_blobs.tgz",
     "https://motional-nuscenes.s3.amazonaws.com/public/v1.0/v1.0-trainval08_blobs.tgz"),
    ("v1.0-trainval09_blobs.tgz",
     "https://motional-nuscenes.s3.amazonaws.com/public/v1.0/v1.0-trainval09_blobs.tgz"),
    ("v1.0-trainval10_blobs.tgz",
     "https://motional-nuscenes.s3.amazonaws.com/public/v1.0/v1.0-trainval10_blobs.tgz"),
]

# Check if already fully extracted (skip download+extract if so)
META_DIR    = f"{NUSCENES_ROOT}/v1.0-trainval"
SAMPLES_DIR = f"{NUSCENES_ROOT}/samples"
if os.path.exists(META_DIR) and os.path.exists(SAMPLES_DIR) and not FORCE_RERUN:
    print("✓ nuScenes trainval already extracted — skipping download.")
    print("  Set FORCE_RERUN=True to re-download.")
else:
    for i, (name, url) in enumerate(URLS):
        tarball = f"/content/{name}"
        print(f"\n[{i+1}/{len(URLS)}] Downloading {name}...")
        if not (os.path.exists(tarball) and os.path.getsize(tarball) > 1_000_000):
            !wget -q --show-progress -O {tarball} "{url}"
        if os.path.exists(tarball) and os.path.getsize(tarball) > 1_000_000:
            print(f"  Extracting → {NUSCENES_ROOT}...")
            !tar -xzf {tarball} -C {NUSCENES_ROOT}
            os.remove(tarball)  # Delete immediately to save disk
            print(f"  ✓ Extracted and tarball deleted.")
        else:
            print(f"  ⚠ Download failed or too small — check URL and retry.")
        !df -h /content | tail -1

print("\n✓ nuScenes trainval download complete.")
!ls {NUSCENES_ROOT}

In [ ]:
import os
from nuscenes.nuscenes import NuScenes

NUSCENES_ROOT = f"{DATA_ROOT}/nuscenes"
META_DIR      = f"{NUSCENES_ROOT}/v1.0-trainval"

if not os.path.exists(META_DIR):
    print("⚠ v1.0-trainval/ not found — complete Phase A first.")
else:
    try:
        nusc = NuScenes(version="v1.0-trainval", dataroot=NUSCENES_ROOT, verbose=False)
        n_scenes  = len(nusc.scene)
        n_samples = len(nusc.sample)
        n_annots  = len(nusc.sample_annotation)
        print(f"✓ nuScenes v1.0-trainval loaded:")
        print(f"  Scenes      : {n_scenes}  (expected 850)")
        print(f"  Samples     : {n_samples}  (expected 28130)")
        print(f"  Annotations : {n_annots}")
        if n_scenes != 850 or n_samples != 28130:
            print("⚠ Unexpected counts — some tarballs may be missing or corrupted.")
    except Exception as e:
        print(f"✗ Load failed: {e}")

## Phase B — DADA-2000 Download

~15 GB accident dashcam dataset from HuggingFace (git lfs).
Set `DOWNLOAD_DADA = False` to skip if already on Drive.

In [ ]:
import os, shutil

DADA_DRIVE = f"{DATA_ROOT}/dada2000"
os.makedirs(DADA_DRIVE, exist_ok=True)

# Possible directory structures to search after cloning
DADA_SEARCH_PATHS = [
    f"{DADA_DRIVE}/DADA-2000",
    f"{DADA_DRIVE}/MM-AU/DADA-2000",
    f"{DADA_DRIVE}/data/DADA-2000",
    f"{DADA_DRIVE}/dataset/DADA-2000",
    DADA_DRIVE,
]

def find_dada_root(search_paths):
    # Return the first directory that looks like DADA-2000 data
    for p in search_paths:
        if os.path.isdir(p) and os.listdir(p):
            # Check for numbered category subdirectories (001/, 002/, ...)
            subdirs = [d for d in os.listdir(p)
                       if os.path.isdir(os.path.join(p, d)) and d.isdigit()]
            if subdirs:
                return p
    return None

existing = find_dada_root(DADA_SEARCH_PATHS)

if not DOWNLOAD_DADA:
    print("DOWNLOAD_DADA=False — skipping DADA-2000 download.")
    if existing:
        print(f"✓ Found existing DADA-2000 data at: {existing}")
    else:
        print("⚠ No DADA-2000 data found. Set DOWNLOAD_DADA=True to download.")
elif existing and not FORCE_RERUN:
    print(f"✓ DADA-2000 already present at: {existing}")
    print("  Set FORCE_RERUN=True to re-download.")
else:
    CLONE_DIR = f"{DADA_DRIVE}/repo_clone"
    if os.path.exists(CLONE_DIR):
        shutil.rmtree(CLONE_DIR)

    # Install git-lfs if needed
    !apt-get install -y git-lfs -q 2>&1 | tail -1
    !git lfs install --skip-smudge -q

    print("Cloning DADA-2000 from HuggingFace (~15 GB via git lfs)...")
    !git clone https://huggingface.co/datasets/DADA-2000/DADA-2000 {CLONE_DIR} 2>&1 | tail -5
    !cd {CLONE_DIR} && git lfs pull 2>&1 | tail -5

    # Locate actual data inside clone
    found = find_dada_root([CLONE_DIR] + [f"{CLONE_DIR}/{sub}"
                            for sub in ("DADA-2000", "data", "dataset", "MM-AU/DADA-2000")])
    if found:
        dest = f"{DADA_DRIVE}/DADA-2000"
        if os.path.exists(dest):
            shutil.rmtree(dest)
        shutil.move(found, dest)
        shutil.rmtree(CLONE_DIR, ignore_errors=True)
        print(f"✓ DADA-2000 data moved to {dest}")
        !ls {dest} | head -10
    else:
        print(f"⚠ Could not locate numbered category dirs inside clone.")
        print(f"  Clone is at {CLONE_DIR} — inspect and move data manually to {DADA_DRIVE}/DADA-2000/")

!df -h /content | tail -1

## Phase C — Rarity Filtering

Scores all 28 K+ nuScenes keyframes on 6 binary signals (proximity, occlusion,
density, weather/night, VRU, cyclist). Keeps frames with score ≥ `MIN_RARITY_SCORE` (3).

Expected output: **500–800 frames** from nuScenes trainval.

`FORCE_RERUN=True` clears and reruns. Output: `metadata.json` immediately
converted to `metadata.jsonl` (required by the unified dataset builder).

In [ ]:
import os, json, shutil

FILTER_OUTPUT = f"{OUTPUTS_ROOT}/data/nuscenes_filtered"
NUSCENES_DIR  = f"{DATA_ROOT}/nuscenes"
json_path     = f"{FILTER_OUTPUT}/metadata.json"
jsonl_path    = f"{FILTER_OUTPUT}/metadata.jsonl"

output_exists = os.path.exists(json_path)
if output_exists and not FORCE_RERUN:
    with open(jsonl_path) as f:
        n = sum(1 for _ in f)
    img_dir = f"{FILTER_OUTPUT}/images"
    n_imgs = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    print(f"✓ Already filtered: {n} frames, {n_imgs} images. Set FORCE_RERUN=True to rebuild.")
else:
    if os.path.exists(FILTER_OUTPUT):
        shutil.rmtree(FILTER_OUTPUT)
    os.makedirs(FILTER_OUTPUT, exist_ok=True)
    os.chdir(REPO_ROOT)
    print(f"Running nuScenes rarity filter (version={NUSCENES_VERSION}, min_score={MIN_RARITY_SCORE})...")
    !python scripts/run_nuscenes_filter.py \
        --nuscenes-root {NUSCENES_DIR} \
        --version {NUSCENES_VERSION} \
        --output-dir {FILTER_OUTPUT} \
        --min-score {MIN_RARITY_SCORE} \
        2>&1

    # CRITICAL: convert JSON array → JSONL (unified builder expects JSONL)
    if os.path.exists(json_path) and not os.path.exists(jsonl_path):
        with open(json_path) as f:
            frames = json.load(f)
        with open(jsonl_path, "w") as f:
            for frame in frames:
                f.write(json.dumps(frame) + "\n")
        print(f"  ✓ Converted {len(frames)} frames → metadata.jsonl")

if os.path.exists(jsonl_path):
    with open(jsonl_path) as f:
        n = sum(1 for _ in f)
    img_dir = f"{FILTER_OUTPUT}/images"
    n_imgs = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    print(f"  ✓ metadata.jsonl : {n} frames")
    print(f"  ✓ images/        : {n_imgs} files")
else:
    print("  ⚠ metadata.json not produced — check filter output above")

!ls -lh {FILTER_OUTPUT} 2>/dev/null

## Phase D — DADA-2000 Keyframe Extraction

Extracts the critical accident moment + 2 context frames from each clip.
Resizes to 672×448. Expected output: **200–400 frames**.

Skipped automatically if DADA-2000 data was not downloaded.

In [ ]:
import os, shutil

DADA_DRIVE  = f"{DATA_ROOT}/dada2000"
DADA_OUTPUT = f"{OUTPUTS_ROOT}/data/dada_extracted"

# Search all possible DADA-2000 directory structures
DADA_SEARCH_PATHS = [
    f"{DADA_DRIVE}/DADA-2000",
    f"{DADA_DRIVE}/MM-AU/DADA-2000",
    f"{DADA_DRIVE}/data/DADA-2000",
    f"{DADA_DRIVE}/dataset/DADA-2000",
    DADA_DRIVE,
]

def find_dada_root(paths):
    for p in paths:
        if os.path.isdir(p) and os.listdir(p):
            subdirs = [d for d in os.listdir(p)
                       if os.path.isdir(os.path.join(p, d)) and d.isdigit()]
            if subdirs:
                return p
    return None

DADA_ROOT = find_dada_root(DADA_SEARCH_PATHS)

if not DADA_ROOT:
    print("⚠ DADA-2000 data not found — skipping extraction (nuScenes-only run).")
    print(f"  Searched: {DADA_SEARCH_PATHS}")
else:
    print(f"✓ DADA-2000 found at: {DADA_ROOT}")
    output_exists = os.path.exists(f"{DADA_OUTPUT}/metadata.jsonl")
    if output_exists and not FORCE_RERUN:
        with open(f"{DADA_OUTPUT}/metadata.jsonl") as f:
            n = sum(1 for _ in f)
        print(f"✓ Already extracted: {n} frames. Set FORCE_RERUN=True to rebuild.")
    else:
        if os.path.exists(DADA_OUTPUT):
            shutil.rmtree(DADA_OUTPUT)
        os.makedirs(DADA_OUTPUT, exist_ok=True)
        os.chdir(REPO_ROOT)
        print("Running DADA-2000 extraction...")
        !python scripts/run_dada_extraction.py \
            --dada-root {DADA_DRIVE} \
            --output-dir {DADA_OUTPUT} \
            2>&1
        if os.path.exists(f"{DADA_OUTPUT}/metadata.jsonl"):
            with open(f"{DADA_OUTPUT}/metadata.jsonl") as f:
                n = sum(1 for _ in f)
            print(f"✓ DADA-2000 extraction complete: {n} frames")
        else:
            print("⚠ metadata.jsonl not found — check output above")
    !ls {DADA_OUTPUT}

## Phase E — PySpark ETL + Unified Dataset

**PySpark**: distributed rarity scoring + analytics over the filtered nuScenes frames.

**Unified dataset**: merges nuScenes + DADA-2000 (if available) into stratified
train/val/test splits, then combines all splits into `all_manifest.jsonl` for
single-pass annotation (annotating per-split causes overwrite bugs).

In [ ]:
import os, shutil

SPARK_OUTPUT = f"{OUTPUTS_ROOT}/data/spark_processed"
spark_exists = (os.path.exists(SPARK_OUTPUT) and
                bool(os.listdir(SPARK_OUTPUT))) if os.path.exists(SPARK_OUTPUT) else False

if spark_exists and not FORCE_RERUN:
    print(f"✓ Spark output already exists. Set FORCE_RERUN=True to rebuild.")
    !ls {SPARK_OUTPUT}
else:
    if os.path.exists(SPARK_OUTPUT):
        shutil.rmtree(SPARK_OUTPUT)
    os.makedirs(SPARK_OUTPUT, exist_ok=True)
    os.environ.setdefault("JAVA_HOME", "/usr/lib/jvm/java-11-openjdk-amd64")
    os.chdir(REPO_ROOT)
    print("Running PySpark rarity scoring pipeline...")
    !python scripts/run_spark_pipeline.py \
        --version {NUSCENES_VERSION} \
        --nuscenes-root {DATA_ROOT}/nuscenes \
        --output-dir {SPARK_OUTPUT} \
        2>&1
    print("✓ Spark pipeline complete.")
    !ls {SPARK_OUTPUT}

In [ ]:
import os, json, shutil

FILTER_OUTPUT  = f"{OUTPUTS_ROOT}/data/nuscenes_filtered"
DADA_OUTPUT    = f"{OUTPUTS_ROOT}/data/dada_extracted"
UNIFIED_OUTPUT = f"{OUTPUTS_ROOT}/data/unified"
ALL_MANIFEST   = f"{UNIFIED_OUTPUT}/all_manifest.jsonl"

output_exists = os.path.exists(f"{UNIFIED_OUTPUT}/train_manifest.jsonl")
if output_exists and not FORCE_RERUN:
    print("✓ Unified dataset already built. Set FORCE_RERUN=True to rebuild.")
    !ls {UNIFIED_OUTPUT}
else:
    if os.path.exists(UNIFIED_OUTPUT):
        shutil.rmtree(UNIFIED_OUTPUT)
    os.makedirs(UNIFIED_OUTPUT, exist_ok=True)

    HAS_DADA  = os.path.exists(f"{DADA_OUTPUT}/metadata.jsonl")
    dada_flag = f"--dada-dir {DADA_OUTPUT}" if HAS_DADA else "--nuscenes-only"
    src_desc  = "nuScenes trainval + DADA-2000" if HAS_DADA else "nuScenes trainval only"
    os.chdir(REPO_ROOT)
    print(f"Building unified dataset ({src_desc})...")
    !python scripts/run_build_unified_dataset.py \
        --nuscenes-dir {FILTER_OUTPUT} \
        --output-dir {UNIFIED_OUTPUT} \
        {dada_flag} \
        2>&1

    print("\n✓ Split manifests:")
    for split in ("train", "val", "test"):
        p = f"{UNIFIED_OUTPUT}/{split}_manifest.jsonl"
        if os.path.exists(p):
            with open(p) as f:
                n = sum(1 for _ in f)
            print(f"  {split}: {n} frames")
        else:
            print(f"  {split}: NOT FOUND")

# Always rebuild all_manifest.jsonl — cheap, required by annotation pipeline.
# Annotating per-split would overwrite the shared annotated_manifest.json each time.
print("\nBuilding all_manifest.jsonl (single-pass annotation input)...")
combined = []
for split in ("train", "val", "test"):
    p = f"{UNIFIED_OUTPUT}/{split}_manifest.jsonl"
    if os.path.exists(p):
        with open(p) as f:
            for line in f:
                frame = json.loads(line)
                frame["split"] = split
                combined.append(frame)
with open(ALL_MANIFEST, "w") as f:
    for frame in combined:
        f.write(json.dumps(frame) + "\n")
print(f"  ✓ Combined {len(combined)} frames → all_manifest.jsonl")

## Phase F — LLM Annotation

`USE_MOCK_LLM=False` (default for full-scale): real Claude Vision API, ~$15–20 for 700–1,200 frames.
Store your API key in **Colab Secrets** (key icon in sidebar) as `ANTHROPIC_API_KEY`.

`USE_MOCK_LLM=True`: free mock annotations (random but structurally valid), no API key needed.

`FORCE_RERUN=True`: clears previous annotations and reruns from scratch (cache supports resume).

In [ ]:
import os, shutil, json

UNIFIED_OUTPUT    = f"{OUTPUTS_ROOT}/data/unified"
ANNOTATION_OUTPUT = f"{OUTPUTS_ROOT}/data/annotated"
ALL_MANIFEST      = f"{UNIFIED_OUTPUT}/all_manifest.jsonl"

output_exists = os.path.exists(f"{ANNOTATION_OUTPUT}/annotated_manifest.json")
if output_exists and not FORCE_RERUN:
    with open(f"{ANNOTATION_OUTPUT}/annotated_manifest.json") as f:
        n = len(json.load(f))
    print(f"✓ Annotation already done ({n} frames). Set FORCE_RERUN=True to rerun.")
else:
    if os.path.exists(ANNOTATION_OUTPUT):
        shutil.rmtree(ANNOTATION_OUTPUT)
    os.makedirs(ANNOTATION_OUTPUT, exist_ok=True)
    os.chdir(REPO_ROOT)

    if USE_MOCK_LLM:
        print("Running annotation (MOCK mode — no API key needed)...")
        !python scripts/run_annotation_pipeline.py \
            --manifest {ALL_MANIFEST} \
            --output-dir {ANNOTATION_OUTPUT} \
            --mock-llm \
            2>&1
    else:
        print("Running annotation (REAL Claude API — charges apply)...")
        try:
            from google.colab import userdata
            os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
            print("✓ ANTHROPIC_API_KEY loaded from Colab Secrets")
        except Exception:
            print("⚠ Could not load API key from Colab Secrets.")
            print("  Add it via the key icon in the sidebar, then rerun this cell.")
            print("  Or set manually: os.environ[\"ANTHROPIC_API_KEY\"] = \"sk-ant-...\"")
            raise
        !python scripts/run_annotation_pipeline.py \
            --manifest {ALL_MANIFEST} \
            --output-dir {ANNOTATION_OUTPUT} \
            2>&1

# Print diversity stats
ann_path = f"{ANNOTATION_OUTPUT}/annotated_manifest.json"
if os.path.exists(ann_path):
    with open(ann_path) as f:
        annotated = json.load(f)
    labels, severities, sources = {}, {}, {}
    for frame in annotated:
        src = frame.get("source", "unknown")
        sources[src] = sources.get(src, 0) + 1
        for h in frame.get("annotations", {}).get("hazards", []):
            lbl = h.get("label", "")
            sev = str(h.get("severity", ""))
            labels[lbl] = labels.get(lbl, 0) + 1
            severities[sev] = severities.get(sev, 0) + 1
    print(f"\n✓ Annotation complete: {len(annotated)} frames")
    print(f"  Sources    : {sources}")
    print(f"  Top labels : {sorted(labels.items(), key=lambda x: -x[1])[:6]}")
    print(f"  Severities : {sorted(severities.items())}")
else:
    print("⚠ annotated_manifest.json not found — check output above")

## Phase G — SFT JSONL Creation

Converts annotations to Qwen3-VL chat-format JSONL with **correct absolute image paths**.

Image paths are resolved from `metadata.json` using `frame_id → exported_image_path`
(with fallback to `cam_front_path` basename). Always clears and rebuilds SFT files
to ensure paths are up-to-date.

In [ ]:
import json, os, shutil

ANNOTATION_OUTPUT = f"{OUTPUTS_ROOT}/data/annotated"
SFT_OUTPUT = f"{OUTPUTS_ROOT}/data/sft_ready"
FILTER_META = f"{OUTPUTS_ROOT}/data/nuscenes_filtered/metadata.json"
IMAGES_DIR = f"{OUTPUTS_ROOT}/data/nuscenes_filtered/images"

if os.path.exists(SFT_OUTPUT):
    shutil.rmtree(SFT_OUTPUT)
os.makedirs(SFT_OUTPUT, exist_ok=True)

with open(FILTER_META) as f:
    filtered = json.load(f)
id_to_image = {}
for frame in filtered:
    fid = frame.get("sample_token", "")
    img = frame.get("exported_image_path", "")
    if img and not os.path.isabs(img):
        img = f"{OUTPUTS_ROOT}/data/nuscenes_filtered/{img}"
    if not img or not os.path.exists(img):
        cam_path = frame.get("cam_front_path", "")
        if cam_path:
            candidate = f"{IMAGES_DIR}/{os.path.basename(cam_path)}"
            if os.path.exists(candidate):
                img = candidate
    id_to_image[fid] = img
mapped = sum(1 for v in id_to_image.values() if v)
print(f"Image mappings: {mapped}/{len(id_to_image)}")

annot_path = f"{ANNOTATION_OUTPUT}/annotated_manifest.json"
assert os.path.exists(annot_path), f"No annotations at {annot_path}"
with open(annot_path) as f:
    annotated = json.load(f)
print(f"Annotated frames: {len(annotated)}")

labels = set()
severities = set()
for frame in annotated:
    for h in frame.get("annotations", {}).get("hazards", []):
        labels.add(h.get("label", ""))
        severities.add(h.get("severity", ""))
print(f"Labels: {labels}")
print(f"Severities: {severities}")

SYSTEM_PROMPT = "You are DriveSense, an autonomous vehicle hazard detection system. Analyze the dashcam image and identify all safety-critical hazards. Output a structured JSON response with bounding boxes (normalized 0-1000), hazard classification, severity assessment, reasoning, and recommended action."
USER_PROMPT = "Analyze this dashcam image for safety hazards. Identify all hazards with bounding boxes, classify each hazard, assess severity, explain your reasoning, and recommend an action. Respond with JSON only."

for split in ["train", "val", "test"]:
    split_frames = [f for f in annotated if f.get("split") == split and f.get("annotations")]
    sft_path = f"{SFT_OUTPUT}/sft_{split}.jsonl"
    with open(sft_path, 'w') as out:
        for frame in split_frames:
            fid = frame.get("frame_id", "")
            img_path = id_to_image.get(fid, frame.get("image_path", ""))
            example = {
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": [
                        {"type": "image", "image": img_path},
                        {"type": "text", "text": USER_PROMPT}
                    ]},
                    {"role": "assistant", "content": json.dumps(frame["annotations"])}
                ],
                "frame_id": fid,
                "source": frame.get("source", ""),
                "split": split,
            }
            out.write(json.dumps(example) + '\n')
    with open(sft_path) as f:
        count = sum(1 for _ in f)
    size = os.path.getsize(sft_path) / 1024
    print(f"  ✓ sft_{split}.jsonl: {count} examples ({size:.1f} KB)")

with open(f"{SFT_OUTPUT}/sft_train.jsonl") as f:
    ex = json.loads(f.readline())
for msg in ex["messages"]:
    if isinstance(msg.get("content"), list):
        for item in msg["content"]:
            if item.get("type") == "image":
                p = item["image"]
                print(f"\nImage path: {p}")
                print(f"Exists: {os.path.exists(p)}")
                assert os.path.exists(p), "Image not found!"
print(f"\n✅ SFT data ready at {SFT_OUTPUT}")

In [ ]:
import os, shutil

# Delete ephemeral download directory to free SSD space before training
if os.path.exists(LOCAL_DATA):
    shutil.rmtree(LOCAL_DATA)
    print(f"✓ Deleted {LOCAL_DATA} (ephemeral download cache)")

print("\nDisk space after cleanup:")
!df -h /content | tail -1
!df -h {PROJECT_ROOT} | tail -1

print("\nOutputs on Drive (persistent):")
!du -sh {OUTPUTS_ROOT}/data/* 2>/dev/null

## Verification

Check all pipeline outputs and print final diversity statistics.

In [ ]:
import os, json

SFT_DIR     = f"{OUTPUTS_ROOT}/data/sft_ready"
FILTER_OUT  = f"{OUTPUTS_ROOT}/data/nuscenes_filtered"
UNIFIED_OUT = f"{OUTPUTS_ROOT}/data/unified"
ANNOTATED   = f"{OUTPUTS_ROOT}/data/annotated"

checks = {
    "nuScenes filtered (JSON)":  f"{FILTER_OUT}/metadata.json",
    "nuScenes filtered (JSONL)": f"{FILTER_OUT}/metadata.jsonl",
    "Spark processed":           f"{OUTPUTS_ROOT}/data/spark_processed",
    "Unified train":             f"{UNIFIED_OUT}/train_manifest.jsonl",
    "Unified val":               f"{UNIFIED_OUT}/val_manifest.jsonl",
    "Unified test":              f"{UNIFIED_OUT}/test_manifest.jsonl",
    "All manifest":              f"{UNIFIED_OUT}/all_manifest.jsonl",
    "Annotated manifest":        f"{ANNOTATED}/annotated_manifest.json",
    "SFT train":                 f"{SFT_DIR}/sft_train.jsonl",
    "SFT val":                   f"{SFT_DIR}/sft_val.jsonl",
    "SFT test":                  f"{SFT_DIR}/sft_test.jsonl",
}

print("=" * 58)
print("  Pipeline Verification — Full Scale")
print("=" * 58)
for label, path in checks.items():
    ok = os.path.exists(path)
    if ok and path.endswith(".jsonl"):
        with open(path) as f:
            n = sum(1 for _ in f)
        detail = f" ({n} records)"
    elif ok and path.endswith(".json"):
        detail = f" ({os.path.getsize(path)//1024} KB)"
    elif ok:
        detail = ""
    else:
        detail = ""
    print(f"  {'✓' if ok else '✗'}  {label:<30}{detail}")
print("=" * 58)

# Diversity stats from SFT train
sft_train = f"{SFT_DIR}/sft_train.jsonl"
if os.path.exists(sft_train):
    label_counts, sev_counts, source_counts = {}, {}, {}
    with open(sft_train) as f:
        for line in f:
            ex = json.loads(line)
            source_counts[ex.get("source", "?")] = source_counts.get(ex.get("source", "?"), 0) + 1
            for msg in ex.get("messages", []):
                if msg.get("role") == "assistant":
                    try:
                        ann = json.loads(msg["content"])
                        for h in ann.get("hazards", []):
                            lbl = h.get("label", "")
                            sev = str(h.get("severity", ""))
                            label_counts[lbl] = label_counts.get(lbl, 0) + 1
                            sev_counts[sev] = sev_counts.get(sev, 0) + 1
                    except Exception:
                        pass
    print("\n  Diversity (sft_train.jsonl):")
    print(f"    Sources    : {source_counts}")
    top_labels = sorted(label_counts.items(), key=lambda x: -x[1])[:8]
    print(f"    Top labels : {top_labels}")
    print(f"    Severities : {sorted(sev_counts.items())}")

total_sft = 0
for split in ("train", "val", "test"):
    p = f"{SFT_DIR}/sft_{split}.jsonl"
    if os.path.exists(p):
        with open(p) as f:
            n = sum(1 for _ in f)
        total_sft += n
print(f"\n  Total SFT examples : {total_sft}")
if 700 <= total_sft <= 1200:
    print("  ✅ Target met: 700–1,200 high-quality examples")
elif total_sft > 0:
    print(f"  ⚠ Outside target range 700–1,200 (got {total_sft})")
print("\nReady for Notebook 01 (Training)")